# Unified Entra ID (Azure AD) Auth: Data Access Samples for Azure ML

This notebook demonstrates how to **consume data from multiple data sources using Microsoft Entra ID (Azure AD) tokens and RBAC**, designed to run seamlessly on **Azure Machine Learning compute** (Managed Identity) or locally (Azure CLI sign-in / Service Principal).

**Covered sources**:
- Azure SQL Database / SQL Managed Instance (AAD access token via `pyodbc`)
- Azure Data Explorer (Kusto)
- Azure Blob Storage / ADLS Gen2 (RBAC)
- Azure Cosmos DB (SQL API)
- Power BI REST API (datasets & queries)
- Generic Entra-protected REST API (custom scope)

> ⚠️ **Prereqs**: Grant the RBAC roles noted in each section to the identity used here (Managed Identity, Service Principal, or your user if developing locally). Ensure network connectivity (VNet/Private Endpoints) where applicable.

## 0. Install required packages
Run this once per environment. In Azure ML notebooks, `%pip` is supported. If you're on a locked-down network, you may need an approved index or whitelisted domains.

In [ ]:
%pip install -q azure-identity pandas pyodbc sqlalchemy \
    azure-kusto-data azure-kusto-ingest \
    azure-storage-blob azure-storage-file-datalake \
    azure-cosmos requests

# Note: ODBC Driver 17/18 for SQL Server must be installed at the OS level (not via pip).
# On Ubuntu: sudo ACCEPT_EULA=Y apt-get install -y msodbcsql18 unixodbc-dev


## 1. Configuration & Credentials
This notebook uses `DefaultAzureCredential`, which tries (in order): Managed Identity (when running in Azure), environment credentials (Service Principal), Azure CLI login, and interactive browser (if not excluded).

In [ ]:
import os, sys
from azure.identity import DefaultAzureCredential

# Create a shared credential. In Azure ML compute, this will use the assigned Managed Identity.
credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)

# --- Placeholders: update to your resources ---
AZURE_SQL_SERVER = os.getenv('AZURE_SQL_SERVER', '<your-sql-server>.database.windows.net')
AZURE_SQL_DB = os.getenv('AZURE_SQL_DB', '<your-db>')

KUSTO_CLUSTER_URI = os.getenv('KUSTO_CLUSTER_URI', 'https://<your-cluster>.kusto.windows.net')
KUSTO_DB = os.getenv('KUSTO_DB', '<your-db>')

STORAGE_ACCOUNT = os.getenv('STORAGE_ACCOUNT', '<yourstorageacct>')
BLOB_CONTAINER = os.getenv('BLOB_CONTAINER', 'samples')
BLOB_PATH = os.getenv('BLOB_PATH', 'data/sample.csv')
USE_ADLS = os.getenv('USE_ADLS', 'false').lower() in ('true','1','yes')

COSMOS_ENDPOINT = os.getenv('COSMOS_ENDPOINT', 'https://<your-cosmos-account>.documents.azure.com:443/')
COSMOS_DB = os.getenv('COSMOS_DB', '<your-db>')
COSMOS_CONTAINER = os.getenv('COSMOS_CONTAINER', '<your-container>')

POWERBI_GROUP_ID = os.getenv('POWERBI_GROUP_ID')  # optional
POWERBI_DATASET_ID = os.getenv('POWERBI_DATASET_ID')  # optional

CUSTOM_API_BASE = os.getenv('CUSTOM_API_BASE', 'https://api.contoso.com')
CUSTOM_API_PATH = os.getenv('CUSTOM_API_PATH', '/v1/ping')
CUSTOM_API_SCOPE = os.getenv('CUSTOM_API_SCOPE', 'api://00000000-0000-0000-0000-000000000000/.default')

print('Configuration loaded. Update placeholders or set environment variables as needed.')


### (Optional) Sanity check: acquire a token for a given resource scope
Useful when debugging RBAC/consent issues.

In [ ]:
def get_access_token(scope: str) -> str:
    """Return a short preview of an access token for the provided scope.
    Example scopes:
      - 'https://database.windows.net/.default' (Azure SQL)
      - 'https://kusto.kusto.windows.net/.default' (Kusto)
      - 'https://storage.azure.com/.default' (ADLS)
      - 'https://analysis.windows.net/powerbi/api/.default' (Power BI)
      - 'api://<client-id>/.default' (Custom API)
    """
    token = credential.get_token(scope).token
    return token[:20] + '...' + token[-20:]

print('Example SQL token preview:', get_access_token('https://database.windows.net/.default'))


## 2. Azure SQL Database / Managed Instance (AAD token via pyodbc)
**RBAC/SQL prerequisites**:
1. Configure an **Entra ID admin** on the SQL server.
2. Create an AAD-contained user in the target database and grant data reader:
```sql
CREATE USER [<principal-name>] FROM EXTERNAL PROVIDER;
EXEC sp_addrolemember 'db_datareader', '<principal-name>';
```
3. Ensure network connectivity (Private Endpoint/VNet as needed).

In [ ]:
import pandas as pd
import pyodbc

def read_from_azure_sql(server, database, query, driver='{ODBC Driver 18 for SQL Server}'):
    """Use AAD access token with pyodbc. Requires ODBC Driver 17/18.
    If you see login errors, confirm AAD admin, user mapping, and driver presence.
    """
    # Acquire an AAD token for Azure SQL
    token = credential.get_token('https://database.windows.net/.default').token
    token_bytes = token.encode('utf-16-le')

    conn_str = (
        f'DRIVER={driver};'
        f'SERVER={server};'
        f'DATABASE={database};'
        'Encrypt=yes;TrustServerCertificate=no;'
        'Authentication=ActiveDirectoryAccessToken;'
    )
    SQL_COPT_SS_ACCESS_TOKEN = 1256
    with pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_bytes}) as conn:
        df = pd.read_sql(query, conn)
    return df

# Example (update the placeholders first):
try:
    sql_df = read_from_azure_sql(
        server=AZURE_SQL_SERVER,
        database=AZURE_SQL_DB,
        query='SELECT TOP (10) name, type_desc FROM sys.objects ORDER BY name'
    )
    display(sql_df)
except Exception as e:
    print('Azure SQL error:', e)


## 3. Azure Data Explorer (Kusto)
Grant the identity a **Viewer** (or higher) role on the ADX database. Prefer filtering in KQL to minimize data transfer.

In [ ]:
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder

def read_from_kusto(cluster_uri, database, kql):
    def _get_kusto_token(scopes=['https://kusto.kusto.windows.net/.default']):
        return credential.get_token(*scopes).token
    kcsb = KustoConnectionStringBuilder.with_aad_token_provider(cluster_uri, _get_kusto_token)
    client = KustoClient(kcsb)
    response = client.execute(database, kql)
    df = pd.DataFrame([row.to_dict() for row in response.primary_results[0]])
    return df

# Example (update placeholders/table names):
try:
    kusto_df = read_from_kusto(
        cluster_uri=KUSTO_CLUSTER_URI,
        database=KUSTO_DB,
        kql='MyTable | take 10'
    )
    display(kusto_df)
except Exception as e:
    print('Kusto error:', e)


## 4. Azure Blob Storage / ADLS Gen2 (RBAC)
Assign **Storage Blob Data Reader** (read) or **Contributor** (read/write) at the account/container/path scope.
When using ADLS Gen2 (hierarchical namespace), use the `dfs.core.windows.net` endpoint.

In [ ]:
from io import StringIO
from azure.storage.blob import BlobServiceClient
from azure.storage.filedatalake import DataLakeServiceClient

def read_csv_from_blob(account_name, container, blob_path, as_adls=False):
    if as_adls:
        service = DataLakeServiceClient(
            account_url=f'https://{account_name}.dfs.core.windows.net',
            credential=credential
        )
        file_client = service.get_file_system_client(container).get_file_client(blob_path)
        data = file_client.download_file().readall()
        return pd.read_csv(StringIO(data.decode('utf-8')))
    else:
        blob_service = BlobServiceClient(
            account_url=f'https://{account_name}.blob.core.windows.net',
            credential=credential
        )
        data = blob_service.get_blob_client(container=container, blob=blob_path).download_blob().readall()
        return pd.read_csv(StringIO(data.decode('utf-8')))

# Example:
try:
    blob_df = read_csv_from_blob(
        account_name=STORAGE_ACCOUNT,
        container=BLOB_CONTAINER,
        blob_path=BLOB_PATH,
        as_adls=USE_ADLS
    )
    display(blob_df.head())
except Exception as e:
    print('Blob/ADLS error:', e)


## 5. Azure Cosmos DB (SQL API) with Entra ID
Assign **Cosmos DB Built-in Data Reader** (or Data Contributor) on the target scope. Ensure the account has AAD RBAC enabled for data plane access.

In [ ]:
from azure.cosmos import CosmosClient

def read_from_cosmos_sql(account_endpoint, database_name, container_name, query='SELECT * FROM c OFFSET 0 LIMIT 100'):
    client = CosmosClient(url=account_endpoint, credential=credential)
    db = client.get_database_client(database_name)
    container = db.get_container_client(container_name)
    items = list(container.query_items(query=query, enable_cross_partition_query=True))
    return pd.DataFrame(items)

# Example:
try:
    cosmos_df = read_from_cosmos_sql(
        account_endpoint=COSMOS_ENDPOINT,
        database_name=COSMOS_DB,
        container_name=COSMOS_CONTAINER,
        query='SELECT TOP 10 * FROM c'
    )
    display(cosmos_df.head())
except Exception as e:
    print('Cosmos error:', e)


## 6. Power BI REST API (datasets & DAX queries)
Requires access to the workspace. For DAX execution, ensure the workspace/capacity allows it. OAuth scope used: `https://analysis.windows.net/powerbi/api/.default`.

In [ ]:
import requests

def powerbi_list_datasets(group_id=None):
    token = credential.get_token('https://analysis.windows.net/powerbi/api/.default').token
    headers = {'Authorization': f'Bearer {token}'}
    if group_id:
        url = f'https://api.powerbi.com/v1.0/myorg/groups/{group_id}/datasets'
    else:
        url = 'https://api.powerbi.com/v1.0/myorg/datasets'
    resp = requests.get(url, headers=headers, timeout=60)
    resp.raise_for_status()
    payload = resp.json()
    return pd.DataFrame(payload.get('value', []))

def powerbi_execute_dax_query(dataset_id, dax_query, group_id=None):
    token = credential.get_token('https://analysis.windows.net/powerbi/api/.default').token
    headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
    body = {'queries': [{'query': dax_query}], 'serializerSettings': {'includeNulls': True}}
    if group_id:
        url = f'https://api.powerbi.com/v1.0/myorg/groups/{group_id}/datasets/{dataset_id}/executeQueries'
    else:
        url = f'https://api.powerbi.com/v1.0/myorg/datasets/{dataset_id}/executeQueries'
    resp = requests.post(url, headers=headers, json=body, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    if data.get('results') and data['results'][0].get('tables'):
        table = data['results'][0]['tables'][0]
        cols = [c['name'] for c in table['columns']]
        rows = table['rows']
        return pd.DataFrame(rows, columns=cols)
    return pd.DataFrame()

# Example: list datasets (update POWERBI_GROUP_ID if needed)
try:
    pbi_df = powerbi_list_datasets(group_id=POWERBI_GROUP_ID)
    display(pbi_df.head())
    # Example DAX (uncomment and set dataset/workspace):
    # dax_df = powerbi_execute_dax_query(dataset_id=POWERBI_DATASET_ID, dax_query='EVALUATE TOPN(10, VALUES("Table"["Column"]))', group_id=POWERBI_GROUP_ID)
    # display(dax_df.head())
except Exception as e:
    print('Power BI error:', e)


## 7. Generic Entra-protected REST API (custom scope)
For APIs protected by Entra ID, request a token for the API's audience/scope (e.g., `api://<client-id>/.default`). Ensure tenant admin consent if required.

In [ ]:
def call_custom_aad_api(base_url, relative_path, scope):
    token = credential.get_token(scope).token
    headers = {'Authorization': f'Bearer {token}'}
    url = base_url.rstrip('/') + '/' + relative_path.lstrip('/')
    resp = requests.get(url, headers=headers, timeout=60)
    resp.raise_for_status()
    try:
        return resp.json()
    except Exception:
        return resp.text

# Example:
try:
    custom = call_custom_aad_api(CUSTOM_API_BASE, CUSTOM_API_PATH, CUSTOM_API_SCOPE)
    print('Custom API response:', str(custom)[:300], '...')
except Exception as e:
    print('Custom API error:', e)


## 8. (Optional) Snowflake with Entra OAuth
If your Snowflake account is configured for Entra OAuth, you can connect with an AAD-issued token (audience set by Snowflake security integration).

**High-level steps**:
1. Configure an Enterprise App in Entra and a Snowflake `SECURITY INTEGRATION`.
2. Grant required roles to your identity in Snowflake.
3. Request an OAuth token for Snowflake's audience and connect using `authenticator="oauth"`.

```python
import snowflake.connector
sf_token = '<aad_access_token_for_snowflake>'
conn = snowflake.connector.connect(
    account='<acct>', warehouse='<wh>', database='<db>', schema='<schema>',
    authenticator='oauth', token=sf_token
)
cur = conn.cursor(); cur.execute('SELECT * FROM MY_TABLE LIMIT 10')
df = cur.fetch_pandas_all()
```

## 9. Troubleshooting & RBAC Cheatsheet
- **Azure SQL**: Ensure AAD admin is set on the server. Map the user in the DB and grant `db_datareader`. Use ODBC Driver 17/18.
- **Kusto**: Grant DB-level **Viewer** or higher. Prefer time/column filtering in KQL.
- **Blob/ADLS**: Grant **Storage Blob Data Reader** (or Contributor). For ADLS Gen2, also ensure POSIX ACLs if enforced.
- **Cosmos DB**: Enable data plane RBAC. Assign **Cosmos DB Built-in Data Reader** or **Data Contributor**.
- **Power BI**: The identity must have workspace access. DAX execution requires proper capacity and permissions.
- **Private Endpoints**: Make sure DNS resolves to private IPs from your compute; configure Private DNS Zones / custom resolvers as needed.
- **Tokens**: If you see `Unauthorized` or `Permission` errors, verify you requested the correct **scope** for the service.

> Tip: In Azure ML, prefer **User-Assigned Managed Identity** for stable identities across compute clusters.